In [4]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [5]:
# Load document 

loader = PyPDFLoader("Alchemist.pdf")

docs = loader.load()

In [7]:
len(docs)

136

In [18]:
# text splitter split docs into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=300,separators=["\n\n", "\n", " ", ""])

chunks = text_splitter.split_documents(docs)

In [19]:
len(chunks)

342

In [16]:
# model defining for embedding 
embedding_model = OllamaEmbeddings(model="qwen3-embedding:0.6b")

In [17]:
llm_model = ChatOllama(model="deepseek-r1:1.5b")

In [20]:
# vector store 

vector_store = Chroma.from_documents(chunks,embedding=embedding_model)

In [21]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [23]:
user_query = """how paulo coelho discover his life meaning?"""

In [24]:
retriever_result = retriever.invoke(user_query)

In [32]:
retriever_result = [doc.page_content for doc in retriever_result]

In [33]:
prompt = f"""
    You are a helpful assistant that helps users find information about the book "The Alchemist" by Paulo Coelho. Use the following context to answer the question at the end.
    Context: {retriever_result}
    Question: {user_query}
"""

In [34]:
prompt

'\n    You are a helpful assistant that helps users find information about the book "The Alchemist" by Paulo Coelho. Use the following context to answer the question at the end.\n    Context: [\'bring not only him but others who read this fine book closer to\\nrecognizing and reaching their own inner destinies.”\\n—Charlotte Zolotow , author of If You Listen\\n“Paulo Coelho gives you the inspiration to follow your own dreams\\nby seeing the world through your own eyes and not someone else’ s.”\\n—Lynn Andrews, author of the Medicine W oman series\\n“Nothing is impossible, such is Coelho’ s message, as long as you\\nwish it with all your heart. No other book bears so much hope; small\\nwonder its author became a guru among all those in search of the\\nmeaning of life.”\\n—Focus  (Germany)\\n“The Alchemist  is a truly poetic book.”\\n—Welt am Sonntag  (Germany)\', \'achieved a self-awareness and a spiritual awakening that he later\\ndescribed in The Pilgrimage.\\nPaulo Coelho once said t

In [35]:
result = llm_model.invoke([HumanMessage(content=prompt)])

In [36]:
print(result.content)

Paulo Coelho discovered his personal meaning through diverse avenues:

1. **Exploring Life in Multiple Means**: In "The Alchemist," he explored themes of finding purpose through different ideas and experiences, much like learning a foreign language.

2. **Insight from Others' Stories**: Through conversations with characters like Charlotte Zolotow and Lynn Andrews, Paulo saw his own story as part of a larger message or pattern.

3. **Spiritual Growth and Calls**: Being called a "singer of life" in The Pilgrimage suggests he felt spiritual growth through his works, indicating a personal journey toward meaning.

4. **Literary Exploration**: His novels, such as those in *If You Listen*, provided themes of self-discovery that resonated with him, helping him find purpose beyond materialism.

Thus, Paulo Coelho's discovery of his personal meaning was rooted in his explorations in literature, spiritual insights, and introspective reflections.
